NOTE: Humhare features ki scale bahut alag hai. Recency 0-600 din tak ja sakta hai, Monetary_log 4-10 ke beech rehta hai, aur AvgOrderValue sau-hazaar tak. Distance ya gradient par based models (Logistic Regression, KNN, SVM) bade numbers wale column ko zyada importance de dete hain, bas isliye ki uski values bade hain. StandardScaler har column ko is formula se same scale par laata hai:

z = (x - mean) / std

Result: har column ka mean ≈ 0 aur std ≈ 1.

1. Day 63 ke splits load karo:

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

X_train = pd.read_csv('../../data/processed/X_train.csv')
X_test = pd.read_csv('../../data/processed/X_test.csv')
y_train = pd.read_csv('../../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../../data/processed/y_test.csv').squeeze()

print(X_train.shape, X_test.shape)

(4226, 8) (1057, 8)


2. Problem dekho: features ki ranges:

In [2]:
num_cols = ['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts']
print(X_train[num_cols].describe().loc[['mean', 'std', 'min', 'max']].round(2))

      Recency  Frequency  Monetary_log  AvgOrderValue  Tenure  UniqueProducts
mean   207.19       5.61          6.74         374.52  432.19           73.51
std    175.44      10.20          1.36         569.48  180.46           95.40
min      0.00       1.00          0.00           0.00    0.00            1.00
max    648.00     259.00         13.03       14844.77  648.00         1836.00


Max aur std ka fark dekho. Yahi wajah hai ki scaling chahiye.

3. StandardScaler apply karo (sabse zaroori rule: fit sirf train par):

In [3]:
scaler = StandardScaler()
scaler.fit(X_train[num_cols])             # mean aur std sirf train se seekhe

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])   # test par sirf transform

Yahan Region_* dummy columns ko scale nahi kiya. Woh already 0/1 hain.

4. Verify karo:

In [4]:
print("Train mean:\n", X_train_scaled[num_cols].mean().round(3))
print("Train std:\n", X_train_scaled[num_cols].std().round(3))
print("Test mean:\n", X_test_scaled[num_cols].mean().round(3))

Train mean:
 Recency           0.0
Frequency        -0.0
Monetary_log     -0.0
AvgOrderValue     0.0
Tenure            0.0
UniqueProducts    0.0
dtype: float64
Train std:
 Recency           1.0
Frequency         1.0
Monetary_log      1.0
AvgOrderValue     1.0
Tenure            1.0
UniqueProducts    1.0
dtype: float64
Test mean:
 Recency           0.007
Frequency         0.067
Monetary_log     -0.022
AvgOrderValue    -0.047
Tenure           -0.010
UniqueProducts    0.043
dtype: float64


Train mean ≈ 0 aur std ≈ 1 aayega. Test ka mean exactly 0 nahi hoga, aur yeh bilkul theek hai. Yehi saboot hai ki test par train ke numbers use hue. Agar test par bhi alag se fit karte, to test set ki information leak hoti (Day 63 ka rule #2).

5. Scaling ka asar dekho: Logistic Regression ko converge hone mein kitne iterations lage:

In [5]:
from sklearn.linear_model import LogisticRegression

m_raw = LogisticRegression(max_iter=5000).fit(X_train, y_train)
m_scaled = LogisticRegression(max_iter=5000).fit(X_train_scaled, y_train)

print("Iterations (unscaled):", m_raw.n_iter_[0])
print("Iterations (scaled)  :", m_scaled.n_iter_[0])

Iterations (unscaled): 525
Iterations (scaled)  : 23


Scaled version aam taur par kam iterations mein converge hota hai. Yahan accuracy ya recall nahi dekh rahe, kyunki proper evaluation Day 71-74 mein hoga. Aaj sirf scaling ka asar dekhna hai.

6. Kin models ko scaling chahiye?

In [6]:
print("""
SCALING ZAROORI HAI : Logistic Regression, KNN, SVM, Neural Networks, PCA
SCALING ZAROORI NAHI: Decision Tree, Random Forest, XGBoost (yeh sirf thresholds par split karte hain)

Hamare project mein: Logistic Regression ke liye scaled data, Decision Tree/XGBoost ke liye raw bhi chalega.
""")


SCALING ZAROORI HAI : Logistic Regression, KNN, SVM, Neural Networks, PCA
SCALING ZAROORI NAHI: Decision Tree, Random Forest, XGBoost (yeh sirf thresholds par split karte hain)

Hamare project mein: Logistic Regression ke liye scaled data, Decision Tree/XGBoost ke liye raw bhi chalega.



7. Outliers ka dhyan: StandardScaler mean aur std use karta hai, jo outliers se kaafi asar lete hain (Day 48-49 yaad karo). Frequency aur AvgOrderValue skewed ho sakte hain. Alternative:

In [7]:
from sklearn.preprocessing import RobustScaler   # median aur IQR use karta hai

robust = RobustScaler().fit(X_train[num_cols])
print(pd.DataFrame(robust.transform(X_train[num_cols]), columns=num_cols).describe().loc[['mean', 'std']].round(2))

      Recency  Frequency  Monetary_log  AvgOrderValue  Tenure  UniqueProducts
mean     0.16       0.52          0.04           0.41   -0.15            0.43
std      0.64       2.04          0.73           2.40    0.65            1.27


Aaj ke liye StandardScaler hi rakhenge (plan yehi kehta hai). Monetary_log already log-transformed hai, isliye us column par outliers ka asar kam hai.

8. Pipeline ka preview (Day 64B mein zaroori hoga):

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer(
    [('scale', StandardScaler(), num_cols)],
    remainder='passthrough'
)
pipe = Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=5000))])

Kyun: Cross-Validation mein har fold ke andar alag train/validation hota hai. Agar scaler pehle poore data par fit kar diya, to har fold leak karega. Pipeline scaling ko model ke saath har fold ke andar apne aap fit karti hai.

9. Scaler aur scaled data save karo:

In [9]:
import joblib, os
os.makedirs('models', exist_ok=True)
joblib.dump(scaler, 'models/scaler.joblib')

X_train_scaled.to_csv('../../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../../data/processed/X_test_scaled.csv', index=False)

Streamlit app (Day 88-89) mein naye customer ka input bhi isi saved scaler se transform hoga, warna prediction galat aayegi.

practice questions:

1. MinMaxScaler (0-1 range) bhi try karo aur StandardScaler se compare karo: kab ek behtar hota hai, apne shabdon mein likho.

- MinMaxScaler (0 se 1 scale): Yeh saari values ko minimum aur maximum ke beech map kar deta hai jisse sabhi features ki range exact [0, 1] ho jati hai.

    - Kab behtar hota hai? Yeh wahan best hai jahan aapko data ki exact bounds (minimum aur maximum) pata hon ya zaroori hon, jaise Neural Networks (jahan activation functions jaise Sigmoid ke liye bounded input achha hota hai) ya Image Processing (jahan pixel intensity 0 se 255 hoti hai).


- StandardScaler (Mean = 0, Std = 1): Yeh data ko standard normal distribution mein convert karta hai jiska mean 0 aur standard deviation 1 hota hai.

    - Kab behtar hota hai? Yeh Logistic Regression, SVM, aur KNN ke liye zyada reliable maana jata hai, khaaskar tab jab data mein chhote-mote outliers hon. MinMaxScaler outliers ke samne bahut sensitive hota hai (ek bada outlier baaki saari small values ko ekdum chote range mein squish kar deta hai). Is project mein humara data skewed ho sakta hai, isliye StandardScaler ya RobustScaler zyada stable perform karte hain.

2. Ek galat version banao: poore X par scaler fit karke dono splits transform karo. Test mean ab kitna alag dikhta hai, aur isse kya galat ho jaata hai?

Agar aap galti se poore X (train + test dono ko milakar) par scaler fit kar dete hain:

- Test Mean par asar: Test set ka mean 0 ke bilkul kareeb (almost 0) aane lagega. Jabki sahi tarike (sirf train par fit karne) mein test ka mean 0 nahi hota, kyunki test data train se alag hota hai.


- Isse kya galat ho jaata hai? (Data Leakage): Isse Data Leakage ho jata hai. Kyunki jab aap poore X par fit karte hain, toh scaler ne test set ka mean aur std bhi calculate kar liya hota hai. Jab aap train set ko transform karte hain, toh usme test set ki thodi si information (future/unseen data ki knowledge) leak ho chuki hoti hai. Real-world production mein jab aapka model bilkul naye, anjaan customer ka data dekhega, toh wahan yeh "leakage" wala advantage gayab ho jayega, aur model ka accuracy test score live environment mein buri tarah fail (overfitting/poor generalization) ho jayega.